# ML Coding — Day 6: Attention IV (positional schemes)

**~1 hour, vectorized NumPy.** How transformers encode position: sinusoidal, RoPE, and ALiBi. These
are (mostly) parameter-free transforms, so today is forward-pass + a key invariance property, plus a
complex-number tensor trick. No element loops in solutions. Run the **helpers cell first** (`_softmax`).

**Q1** sinusoidal PE `[warmup]` · **Q2** RoPE apply `[medium]` · **Q3** ALiBi bias `[medium]` ·
**Q4** RoPE attention + relative-position property `[hard]` · **Q5** RoPE via complex numbers
`[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np


def _softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

## Q1 — Sinusoidal positional encoding  ·  `[warmup]`

**Paper:** Vaswani et al., 2017 (arXiv:1706.03762). **Why it matters:** the original fixed positional
signal — even dims use `sin`, odd dims use `cos`, with geometrically increasing wavelengths, so the
model can attend by relative offset.

`sinusoidal(seq_len, d) -> (seq_len, d)`: `PE[pos, 2i] = sin(pos / 10000^(2i/d))`,
`PE[pos, 2i+1] = cos(pos / 10000^(2i/d))`. Vectorized (no loops).

In [ ]:
import numpy as np


def sinusoidal(seq_len, d):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    pe = sinusoidal(10, 16)
    assert pe.shape == (10, 16)
    assert np.allclose(pe[0, 0::2], 0.0) and np.allclose(pe[0, 1::2], 1.0)   # pos 0: sin0=0, cos0=1
    assert np.all(np.abs(pe) <= 1.0 + 1e-9)
    print("Q1 OK")

_q1()

## Q2 — RoPE (rotary position embedding)  ·  `[medium]`

**Paper:** Su et al., *RoFormer*, 2021 (arXiv:2104.09864). **Why it matters:** RoPE rotates each
2-D feature pair by an angle proportional to position, so the dot product of two rotated vectors
depends only on their **relative** offset — used in LLaMA, GPT-NeoX, etc.

`rope(x, positions) -> x_rot`, `x` of shape `(..., T, d)` (`d` even), `positions` of length `T`.
For pair `i`, angle `= positions · theta_i` with `theta_i = 10000^(-2i/d)`; rotate `(x_2i, x_2i+1)`:
`x'_2i = x_2i·cos − x_2i+1·sin`, `x'_2i+1 = x_2i·sin + x_2i+1·cos`. No loops.

In [ ]:
def rope(x, positions):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(0)
    x = rng.standard_normal((2, 4, 8))
    out = rope(x, np.arange(4))
    assert out.shape == x.shape
    assert np.allclose(out[:, 0, :], x[:, 0, :])              # position 0 -> no rotation
    for t in range(4):                                        # rotation preserves each pair's norm
        for j in range(4):
            assert np.allclose(np.linalg.norm(x[0, t, 2 * j:2 * j + 2]),
                               np.linalg.norm(out[0, t, 2 * j:2 * j + 2]))
    print("Q2 OK")

_q2()

## Q3 — ALiBi bias  ·  `[medium]`

**Paper:** Press et al., *ALiBi*, 2021 (arXiv:2108.12409). **Why it matters:** no positional embeddings
at all — just add a **linear distance penalty** to the attention scores, with a per-head slope. Cheap,
and extrapolates to longer sequences than trained on.

`alibi_bias(num_heads, T) -> (num_heads, T, T)`: head `h` (1-indexed) has slope `2^(-8h/num_heads)`;
`bias[h, i, j] = -slope_h · (i − j)` for `j <= i` (causal), else `-inf`. No loops.

In [ ]:
def alibi_bias(num_heads, T):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    b = alibi_bias(4, 5)
    assert b.shape == (4, 5, 5)
    assert np.allclose(np.diag(b[0]), 0.0)                    # i == j -> distance 0
    assert np.all(np.isinf(b[0][np.triu_indices(5, 1)]))      # above diagonal masked
    assert b[0, 4, 0] < b[0, 4, 3] < 0                        # farther key -> more negative
    assert b[0, 4, 0] < b[3, 4, 0]                            # bigger slope on head 0
    print("Q3 OK")

_q3()

## Q4 — RoPE attention & the relative-position property  ·  `[hard]`

**Paper:** Su et al., 2021. **Why it matters:** applying RoPE to Q and K *before* the dot product makes
attention scores depend only on `i − j`. Implement `rope_attention(Q, K, V) -> (out, scores)`:
RoPE-rotate Q and K (positions `0..T-1`), scaled dot-product `QᵀK/√d`, softmax, `@V`. Return the
context and the pre-softmax scores.

**The property to verify:** `⟨rope(q, m), rope(k, n)⟩` depends only on `m − n` — shifting both
positions by a constant leaves the score unchanged. No loops.

In [ ]:
def rope_attention(Q, K, V):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(1)
    B, T, d = 2, 5, 8
    Q = rng.standard_normal((B, T, d)); K = rng.standard_normal((B, T, d)); V = rng.standard_normal((B, T, 6))
    out, scores = rope_attention(Q, K, V)
    assert out.shape == (B, T, 6) and scores.shape == (B, T, T)
    assert np.allclose(_softmax(scores).sum(-1), 1.0)
    q, k = rng.standard_normal(d), rng.standard_normal(d)

    def dot_at(m, n):
        qm = rope(q[None, None, :], np.array([m]))[0, 0]
        kn = rope(k[None, None, :], np.array([n]))[0, 0]
        return qm @ kn

    assert np.isclose(dot_at(5, 3), dot_at(7, 5))             # same relative offset (2)
    assert np.isclose(dot_at(2, 0), dot_at(12, 10))
    assert not np.isclose(dot_at(5, 3), dot_at(5, 4))         # different offset
    print("Q4 OK")

_q4()

## Q5 — RoPE via complex numbers  ·  `[hard · tensor trick]`

**Paper:** Su et al., 2021 (this is how the official implementation views it). **The trick:** treat each
feature pair as a **complex number** `z = x_2i + i·x_2i+1` and rotate by multiplying with
`e^{i·angle}` — one complex multiply replaces the 2×2 rotation. Implement `rope_complex(x, positions)`
(same signature/result as `rope`) using `np` complex arithmetic; it must match Q2 exactly.

**Plan:** build `z` of shape `(..., T, d/2)` complex; multiply by `np.exp(1j · angle)`; write
`z.real` back to even indices and `z.imag` to odd. No loops.

In [ ]:
def rope_complex(x, positions):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(2)
    x = rng.standard_normal((3, 6, 8))
    pos = np.arange(6)
    assert np.allclose(rope_complex(x, pos), rope(x, pos))
    print("Q5 OK")

_q5()

## Likely follow-ups
- RoPE long-context tricks: NTK-aware / linear position scaling, YaRN.
- Why ALiBi extrapolates and learned absolute embeddings don't; relative-position bias (T5).
- Applying RoPE only to a fraction of dims (partial rotary); interplay with KV-cache.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def _sm(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def ref_sinusoidal(seq_len, d):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / (10000.0 ** (2 * (i // 2) / d))
    pe = np.zeros((seq_len, d))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe


def ref_rope(x, positions):
    x = np.asarray(x, float)
    d = x.shape[-1]
    half = d // 2
    theta = 10000.0 ** (-2 * np.arange(half) / d)
    ang = np.asarray(positions, float)[:, None] * theta[None, :]    # (T, half)
    cos, sin = np.cos(ang), np.sin(ang)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = np.empty_like(x)
    out[..., 0::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out


def ref_alibi_bias(num_heads, T):
    slopes = 2.0 ** (-8.0 * np.arange(1, num_heads + 1) / num_heads)
    i = np.arange(T)[:, None]
    j = np.arange(T)[None, :]
    dist = i - j
    bias = -slopes[:, None, None] * dist[None, :, :]
    return np.where((j <= i)[None, :, :], bias, -np.inf)


def ref_rope_attention(Q, K, V):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float)
    T, d = Q.shape[-2], Q.shape[-1]
    pos = np.arange(T)
    Qr, Kr = ref_rope(Q, pos), ref_rope(K, pos)
    scores = Qr @ np.swapaxes(Kr, -1, -2) / np.sqrt(d)
    return _sm(scores) @ V, scores


def ref_rope_complex(x, positions):
    x = np.asarray(x, float)
    d = x.shape[-1]
    half = d // 2
    theta = 10000.0 ** (-2 * np.arange(half) / d)
    ang = np.asarray(positions, float)[:, None] * theta[None, :]
    rot = np.exp(1j * ang)
    z = x[..., 0::2] + 1j * x[..., 1::2]
    zr = z * rot
    out = np.empty_like(x)
    out[..., 0::2] = zr.real
    out[..., 1::2] = zr.imag
    return out


_pe = ref_sinusoidal(4, 8); assert _pe.shape == (4, 8) and np.allclose(_pe[0, 1::2], 1.0)
_x = np.random.default_rng(5).standard_normal((1, 3, 4))
assert np.allclose(ref_rope(_x, np.arange(3)), ref_rope_complex(_x, np.arange(3)))
assert ref_alibi_bias(2, 4).shape == (2, 4, 4)
_o, _s = ref_rope_attention(_x, _x, _x); assert _o.shape == (1, 3, 4)
print("reference solutions OK")